# Run reporting

This script demonstrates how to report the optimization process using the AutoIntent library.

In [1]:
from typing import Any

search_space: list[dict[str, Any]] = [
    {
        "node_type": "embedding",
        "target_metric": "retrieval_hit_rate",
        "search_space": [
            {
                "module_name": "retrieval",
                "k": [10],
                "embedder_config": ["avsolatorio/GIST-small-Embedding-v0", "sergeyzh/rubert-tiny-turbo"],
            }
        ],
    },
    {
        "node_type": "scoring",
        "target_metric": "scoring_roc_auc",
        "search_space": [
            {"module_name": "knn", "k": [1, 3, 5, 10], "weights": ["uniform", "distance", "closest"]},
            {"module_name": "linear"},
            {
                "module_name": "dnnc",
                "cross_encoder_config": ["cross-encoder/ms-marco-MiniLM-L6-v2"],
                "k": [1, 3, 5, 10],
            },
        ],
    },
    {
        "node_type": "decision",
        "target_metric": "decision_accuracy",
        "search_space": [{"module_name": "threshold", "thresh": [0.5]}, {"module_name": "argmax"}],
    },
]

### Load Data

Let us use small subset of popular `clinc150` dataset:

In [2]:

from autointent import Dataset

dataset = Dataset.from_hub("DeepPavlov/clinc150_subset")

/home/runner/work/AutoIntent/AutoIntent/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Start Auto Configuration

In [3]:
from autointent import Pipeline

pipeline_optimizer = Pipeline.from_search_space(search_space)

## Reporting

Currently supported reporting options are:
- tensorboard
- wandb

In [4]:
from autointent.configs import LoggingConfig
from pathlib import Path

log_config = LoggingConfig(
    run_name="test_tensorboard", report_to=["tensorboard"], project_dir=Path("my_projects"), dump_modules=False
)

pipeline_optimizer.set_config(log_config)

In [5]:
pipeline_optimizer.fit(dataset)

Memory storage is not compatible with resuming optimization. Modules from previous runs won't be available. Set dump_modules=True in LoggingConfig to enable proper resuming.


/home/runner/work/AutoIntent/AutoIntent/src/autointent/nodes/_node_optimizer.py:82: FutureWarning: `consider_prior` has been deprecated in v4.3.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.3.0.
  sampler_instance = optuna.samplers.TPESampler(
/home/runner/work/AutoIntent/AutoIntent/src/autointent/nodes/_node_optimizer.py:82: FutureWarning: `prior_weight` has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0.
  sampler_instance = optuna.samplers.TPESampler(
Storage directory must be provided for study persistence.


[I 2026-06-22 11:25:22,411] A new study created in memory with name: NodeType.embedding


Storage directory must be provided for study persistence.


/home/runner/work/AutoIntent/AutoIntent/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:2092: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/home/runner/work/AutoIntent/AutoIntent/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:2137: FutureWarning: The default value of the parameter 'scoring' will change from None, i.e. accuracy, to 'neg_log_loss' in version 1.11. To silence this warning, explicitly set the scoring parameter: scoring='neg_log_loss' for the new, scoring='accuracy' or scoring=None for the old default.
  warnings.warn(
/home/runner/work/AutoIntent/AutoIntent/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:2150: FutureWarning: The fitted attributes of LogisticRegressionCV wi

/home/runner/work/AutoIntent/AutoIntent/src/autointent/nodes/_node_optimizer.py:82: FutureWarning: `consider_prior` has been deprecated in v4.3.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.3.0.
  sampler_instance = optuna.samplers.TPESampler(
/home/runner/work/AutoIntent/AutoIntent/src/autointent/nodes/_node_optimizer.py:82: FutureWarning: `prior_weight` has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0.
  sampler_instance = optuna.samplers.TPESampler(
Storage directory must be provided for study persistence.


"argmax" is NOT designed to handle OOS samples, but your data contains it. So, using this method reduces the power of classification.


/home/runner/work/AutoIntent/AutoIntent/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Now results of the optimization process can be viewed in the tensorboard.

```bash
tensorboard --logdir test_tensorboard
```